# Identidades numéricas almacenadas en RB-1521

**Dataset:** RB-1521 · **Cuaderno:** 03 · **Tema:** qué relaciones cierran con las celdas y cuáles no

## Introducción

El `.xls` guarda valores. `xlrd` no entrega una fórmula en estas celdas: entrega el número que Excel dejó escrito. Una identidad útil para HUC es una relación que reproduce esas cifras. Una relación que solo se parece en un día no se implementa.

Se contrastan tres candidatas que el nombre de las columnas sugiere, sin traer ecuaciones de BES-jun:

1. \(Q_{agua} = Q_{fluidos}\times \mathrm{BSW}\) y \(Q_{petroleo} = Q_{fluidos} - Q_{agua}\).
2. \(\mathrm{IP} = Q_{fluidos} / (P_{yac} - \mathrm{PWf})\).
3. \(P_{SUT} = \sqrt{3}\, V\, I\, \mathrm{FP} / 1000\) con factores de potencia 0,80, 0,85, 0,90 y 1.

## Objetivos

Contar en cuántas filas cada candidata reproduce el archivo, y conservar solo la que cierra dentro de una tolerancia de \(10^{-9}\).

## Metodología

La tolerancia absoluta de \(10^{-9}\) separa el redondeo binario de una discrepancia de caudal. No se ajustan coeficientes para reducir el error. Si la diferencia máxima es de decenas o cientos, la candidata se rechaza.


In [1]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from rb1521_lectura import fecha_iso, leer_serie
from rb1521_particion import identidad_exacta, particionar
serie = leer_serie()
print("filas", len(serie), "partición exacta", sum(identidad_exacta(f) for f in serie))
for fila in serie:
    if not identidad_exacta(fila):
        agua, pet = particionar(fila["Q-fluidos"], fila["BSW"])
        print("excepción", fecha_iso(fila["Fecha"]), "Qw archivo", fila["Q-water"], "Qw producto", agua)
errores_ip = []
for fila in serie:
    denominador = fila["P-Yac"] - fila["PWf"]
    errores_ip.append(abs(fila["IP"] - fila["Q-fluidos"] / denominador))
print("error máximo IP", max(errores_ip))
for fp in (0.8, 0.85, 0.9, 1.0):
    err = max(abs(f["P-SUT"] - (3 ** 0.5) * f["VOLT"] * f["AMPs"] * fp / 1000) for f in serie)
    print("error máximo P-SUT con FP", fp, err)


filas 129 partición exacta 128
excepción 2020-07-09 Qw archivo 8522.984 Qw producto 8717.226
error máximo IP 1.4881421086895905
error máximo P-SUT con FP 0.8 102.01067770466658
error máximo P-SUT con FP 0.85 97.79687086585156
error máximo P-SUT con FP 0.9 93.58306402703653
error máximo P-SUT con FP 1.0 85.15545034940646


## Resultados

La partición de caudal reproduce Q-water y Q-oil en 128 de 129 filas. La única excepción es el 2020-07-09: el archivo guarda Q-water = 8522,984 y el producto \(8841 \times 0{,}986\) es 8717,226. No se corrige la celda.

El índice de productividad definido como caudal dividido por el contraste de presiones no reproduce IP. El error máximo supera 1 unidad de la columna. La potencia trifásica con los cuatro factores de potencia ensayados deja errores de decenas en P-SUT. Esas dos expresiones no entran al motor.

## Interpretación

HUC puede repartir un caudal y un BSW con la misma relación que el archivo usa en casi toda la serie. No puede, con este libro, reconstruir IP ni P-SUT. Tampoco hay en el archivo una curva de head de la S11000N contra la cual clasificar el punto como se clasifica un pozo de BES-jun.

## Conclusión

El cálculo reproducible es la partición. El día 2020-07-09 queda como discrepancia registrada, no como un valor a forzar.
